In [ ]:
"""
SCRIPT 01
Persiapan data FKTP untuk Maluku dan Maluku Utara.
Input:
    Data Kementrian Koordinator dan FKTP/
    Data_FKTP_Selindo_Magang.xlsx - Sheet1.csv
Output:
    Output/FKTP.csv
"""
from pathlib import Path
import pandas as pd
# ============================================================
# PATH
# ============================================================
PROJECT_ROOT = Path.cwd().parent  # run from the Script/ folder
DATA_DIR = PROJECT_ROOT / "Data Kementrian Koordinator dan FKTP"
OUTPUT_DIR = PROJECT_ROOT / "Output"
OUTPUT_DIR.mkdir(
    exist_ok=True
)
SRC = DATA_DIR / (
    "Data_FKTP_Selindo_Magang.xlsx - Sheet1.csv"
)
OUT = OUTPUT_DIR / "FKTP.csv"
# ============================================================
# PROVINSI SCOPE
# ============================================================
PROV = {
    "MALUKU",
    "MALUKU UTARA"
}
# ============================================================
# MAIN
# ============================================================
def main():
    # --------------------------------------------------------
    # 1. BACA DATA
    # --------------------------------------------------------
    df = pd.read_csv(
        SRC
    )
    print(
        "Data FKTP berhasil dibaca."
    )
    print(
        f"Total data awal: {len(df)}"
    )
    # --------------------------------------------------------
    # 2. NORMALISASI NAMA PROVINSI
    # --------------------------------------------------------
    df["nmprov_ppk"] = (
        df["nmprov_ppk"]
        .astype(str)
        .str.upper()
        .str.strip()
    )
    # --------------------------------------------------------
    # 3. FILTER PROVINSI
    # --------------------------------------------------------
    df = df[

        df["nmprov_ppk"].isin(PROV)
    ]
    print()
    print(
        "FKTP Maluku + Maluku Utara: "
        f"{len(df)}"
    )
    # --------------------------------------------------------
    # 4. PILIH KOLOM DAN RENAME
    # --------------------------------------------------------
    out = df.rename(
        columns={
            "longitude":
                "Longitude",
            "latitude":
                "Latitude"
        }
    )[
        [
            "kode_ppk",
            "nmppk",
            "nmprov_ppk",
            "nmdati2_ppk",
            "nmkec",
            "Longitude",
            "Latitude"
        ]
    ].copy()
    # --------------------------------------------------------
    # 5. PERBAIKI FORMAT DESIMAL
    # --------------------------------------------------------
    # Data menggunakan koma sebagai desimal:
    # 128,293
    # diubah menjadi:
    # 128.293
    out["Longitude"] = (
        out["Longitude"]
        .astype(str)
        .str.replace(
            ",",
            ".",
            regex=False
        )
    )
    out["Latitude"] = (
        out["Latitude"]
        .astype(str)
        .str.replace(
            ",",
            ".",
            regex=False
        )
    )
    # --------------------------------------------------------
    # 6. KONVERSI KE NUMERIC
    # --------------------------------------------------------
    out["Longitude"] = pd.to_numeric(
        out["Longitude"],
        errors="coerce"
    )
    out["Latitude"] = pd.to_numeric(
        out["Latitude"],
        errors="coerce"
    )
    # --------------------------------------------------------
    # 7. HAPUS KOORDINAT INVALID
    # --------------------------------------------------------
    n0 = len(out)
    out = out.dropna(
        subset=[
            "Longitude",
            "Latitude"
        ]
    )
    out = out[
        (out["Longitude"] != 0)
        &
        (out["Latitude"] != 0)
    ]
    # --------------------------------------------------------
    # 8. SIMPAN OUTPUT
    # --------------------------------------------------------
    out.to_csv(
        OUT,
        index=False,
        encoding="utf-8"
    )
    # --------------------------------------------------------
    # 9. INFORMASI HASIL
    # --------------------------------------------------------
    print()
    print(

        f"FKTP sebelum validasi : {n0}"
    )
    print(

        f"FKTP koordinat valid  : {len(out)}"
    )
    print(
        f"FKTP dibuang           : "
        f"{n0 - len(out)}"
    )
    print()
    print(
        "Contoh koordinat hasil:"
    )
    print(
        out[
            [
                "Longitude",
                "Latitude"
            ]
        ]
        .head()
        .to_string(
            index=False
        )
    )
    print()
    print(
        "Output tersimpan di:"
    )
    print(
        OUT
    )
# ============================================================
# RUN
# ============================================================
if __name__ == "__main__":
    main()